# **SUBSCRIBER**

In [ ]:
import json
import os
import csv
import time
import uuid
from paho.mqtt import client as mqtt_client


# ============================================================
# TODO: UPDATE FOLDER PATH and OUTPUT FILE-NAME IN HERE
# ============================================================
SAVE_FOLDER = "data"  # Update this to your desired folder path
CSV_FILENAME = "test.csv"  # Update this to your desired CSV filename
OUTPUT_CSV = os.path.join(SAVE_FOLDER, CSV_FILENAME)

STATE = {
    "writer_initialized": False,
    "fieldnames": None,
    "first_message_received": False,
    "last_msg_time": None,
    "seen_timestamps": set(),
}

# ============================================================
# TODO: MQTT CONFIG 
# ============================================================
MQTT_CONFIG = {
    "BROKER": "192.168.1.1", 
    "PORT": 1883,
    "TOPIC": "idt/pt/grh/pbtest",
    "CLIENT_ID": f"3B-{uuid.uuid4().hex[:8]}",
    "KEEPALIVE": 120,
}

def connect_mqtt():
    client = mqtt_client.Client(mqtt_client.CallbackAPIVersion.VERSION2, MQTT_CONFIG["CLIENT_ID"])
    client.on_connect = on_connect
    client.on_message = on_message
    client.connect(MQTT_CONFIG["BROKER"], MQTT_CONFIG["PORT"], MQTT_CONFIG["KEEPALIVE"])
    return client


def on_connect(client, userdata, flags, reason_code, properties):
    if not reason_code.is_failure:
        print("Connected to MQTT broker.")
        client.subscribe(MQTT_CONFIG["TOPIC"], qos=1)
        print(f"Subscribed to topic: {MQTT_CONFIG['TOPIC']}")
    else:
        print(f"Failed to connect, reason code: {reason_code}")


def load_seen_timestamps():
    if not os.path.isfile(OUTPUT_CSV):
        return
    with open(OUTPUT_CSV, "r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        if STATE["fieldnames"] is None and reader.fieldnames:
            STATE["fieldnames"] = list(reader.fieldnames)
        for row in reader:
            ts = str(row.get("timestamp", "")).strip()
            if ts:
                STATE["seen_timestamps"].add(ts)


def save_data_csv(data: dict):
    if SAVE_FOLDER:
        os.makedirs(SAVE_FOLDER, exist_ok=True)

    if not STATE["writer_initialized"]:
        load_seen_timestamps()
        if STATE["fieldnames"] is None:
            STATE["fieldnames"] = list(data.keys())
        file_exists = os.path.isfile(OUTPUT_CSV)
        if not file_exists:
            with open(OUTPUT_CSV, "a", newline="", encoding="utf-8") as f:
                writer = csv.DictWriter(f, fieldnames=STATE["fieldnames"])
                writer.writeheader()
        STATE["writer_initialized"] = True

    ts = str(data.get("timestamp", "")).strip()
    if ts and ts in STATE["seen_timestamps"]:
        print(f"Duplicate timestamp skipped: {ts}")
        return False

    with open(OUTPUT_CSV, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=STATE["fieldnames"], extrasaction="ignore")
        writer.writerow(data)

    if ts:
        STATE["seen_timestamps"].add(ts)
    return True


def on_message(client, userdata, msg):
    STATE["first_message_received"] = True
    STATE["last_msg_time"] = time.time()

    try:
        payload_str = msg.payload.decode("utf-8")
        data = json.loads(payload_str)
        if save_data_csv(data):
            print(f"Saved row to {OUTPUT_CSV}: {data}")

    except Exception as e:
        print(f"Error processing message: {e}")


def subscriber(IDLE_TIMEOUT = 10):
    client = connect_mqtt()
    print("Starting MQTT Subscribe. (Timeout Activate)")

    client.loop_start()
    try:
        while True:
            if STATE["first_message_received"]:
                idle = time.time() - STATE["last_msg_time"]
                if idle >= IDLE_TIMEOUT:
                    print(
                        f"No Subscribe message for {IDLE_TIMEOUT}s -- stop subscribe process."
                    )
                    break
            time.sleep(0.5)

    except KeyboardInterrupt:
        print("\nStopping subscriber manually...")

    finally:
        client.loop_stop()
        client.disconnect()
        print("Disconnected from MQTT broker.")


if __name__ == "__main__":
    subscriber()


C:\Users\Admin\AppData\Local\Temp\ipykernel_36396\683742965.py:35: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqtt_client.Client(mqtt_client.CallbackAPIVersion.VERSION1, MQTT_CONFIG["CLIENT_ID"])


Starting MQTT Subscribe. (Timeout Activate)
Connected to MQTT broker.
Subscribed to topic: idt/pt/grh/pbtest
Connected to MQTT broker.
Subscribed to topic: idt/pt/grh/pbtest
